<a href="https://colab.research.google.com/github/iqraghori/TMDB-API-Movie-Data-Analysis/blob/main/the_movie_database_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TMDb API Movie Data Analysis**

## Import Libraries

In [61]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from google.colab import userdata

## API Authentication

In [62]:
TMDB_API = userdata.get('TDBM_API')
#url = f"https://api.themoviedb.org/3/movie/popular?api_key={TMDB_API}&page=500"

# **FETCH MOVIE DATA**

In [ ]:
movies = []
for i in range(1, 501):
  url = f"https://api.themoviedb.org/3/movie/popular?api_key={TMDB_API}&page={i}"
  response = requests.get(url)
  data = response.json()
  movies.extend(data['results'])

In [ ]:
movies[0]

## Convert JSON to DataFrame

In [ ]:
df = pd.DataFrame(movies)
df.head(2)
df.shape

# **DATA CLEANING**


In [ ]:
df_movie = df[['id','adult', 'title', 'release_date', 'popularity', 'vote_average', 'vote_count']]
df_movie.head(2)

In [ ]:
df_movie.info()

In [ ]:
df_movie.shape

In [ ]:
#df_movie['id'].value_counts()[lambda x: x>1] returns a seires not  a dataframe
df_movie[df_movie[['id','title']].duplicated()]
df_movie = df_movie.drop_duplicates(subset=['id','title'])

In [ ]:
df_movie.duplicated().sum()

In [ ]:
df_movie.isnull().sum()

In [ ]:
df_movie['adult'] = df_movie['adult'].astype(int)

# **EXPLORATORY DATA ANALYSIS EDA**

# Univariate Analysis

### KDE Plot

In [ ]:
plt.rcParams["figure.figsize"] = (3,3)

sns.kdeplot(df_movie['popularity'],color='red',fill=True)
plt.show()
plt.savefig('popularity.png', dpi=150)

sns.kdeplot(df_movie['vote_average'],color='purple',fill=True)
plt.show()
plt.savefig('vote_average.png', dpi=150)

sns.kdeplot(df_movie['vote_count'],color='green',fill=True)
plt.show()
plt.savefig('vote_count.png', dpi=150)

### Box Plot removing outliers

In [ ]:
sns.boxplot(df_movie['popularity'],color='red',fill=True)
plt.savefig('popularity_boxplot.png', dpi=150)
plt.show()

sns.boxplot(df_movie['vote_average'],color='purple',fill=True)
plt.savefig('vote_average_boxplot.png', dpi=150)
plt.show()

sns.boxplot(df_movie['vote_count'],color='green',fill=True)
plt.savefig('vote_count_boxplot.png', dpi=150)
plt.show()

In [ ]:
Q1 = df_movie['popularity'].quantile(0.25)
Q3 = df_movie['popularity'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df_movie[df_movie['popularity'] < Q3 + 1.5 * IQR]

how much data did we lose?

In [ ]:
removed = len(df_movie) - len(df_clean)
percent = (removed / len(df_movie)) * 100

print(f"Original:  {len(df_movie)} movies")
print(f"Cleaned:   {len(df_clean)} movies")
print(f"Removed:   {removed} outliers ({percent:.1f}%)")

In [ ]:
df_movie.head(1)

# Bivarate Analysis

### Correlation Heatmap

In [ ]:
df_numeric = df_movie.drop(columns=['id','title','release_date'])
df_numeric.corr()
sns.heatmap(df_numeric.corr(), annot=True, cmap='RdBu')
plt.title('Correlation Heatmap of orignal data')
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

High vote_count = more reliable vote_average. Low vote count ratings are misleading.Popularity and vote count move together, but popularity can spike without proportional votes (viral/controversial movies)

Statical Validation

In [ ]:
corr, pvalue = stats.pearsonr(df_movie['vote_count'], df_movie['vote_average'])
print(f"Correlation: {corr:.3f}, P-value: {pvalue:.4f}")

### Scatter Plots

In [ ]:
sns.scatterplot(data=df_movie, x='popularity', y='vote_average',color='red')
plt.title('Popularity vs Vote Average of orignal data')
plt.savefig('popularity_vs_vote_average(scatterPlot).png', dpi=150)
plt.show()

sns.scatterplot(data=df_movie, x='popularity', y='vote_count',color='green')
plt.title('Popularity vs Vote Count of orignal data')
plt.savefig('popularity_vs_vote_count(scatterPlot).png', dpi=150)
plt.show()

The dataset is heavily imbalanced — a small number of movies dominate in both popularity and vote count, while the majority are obscure low-engagement films. This is typical of real-world movie data following a power law distribution.

In [ ]:
sns.scatterplot(data=df_clean, x='popularity', y='vote_average',color='red')
plt.title('Popularity vs Vote Average of cleaned data (outliers removed)')
plt.savefig('popularity_vs_vote_average(scatterPlot) for cleaned data outliers removed.png', dpi=150)
plt.show()

sns.scatterplot(data=df_clean, x='popularity', y='vote_count',color='green')
plt.title('Popularity vs Vote Count of cleaned data (outliers removed)')
plt.savefig('popularity_vs_vote_count(scatterPlot) for cleaned data outliers removed.png', dpi=150)
plt.show()

# **ADVANCED EDA**

### Weighted rating (IMDB-style)

In [ ]:
df_movie.head(1)

In [ ]:
m = df_movie['vote_count'].quantile(0.7)
c = df_movie['vote_average'].mean()

def weighted_rating(x, m=m, c=c):
  v = x['vote_count']
  r = x['vote_average']
  return (v/(v+m) * r) + (m/(m+v) * c)

df_movie['weighted_rating'] = df_movie.apply(weighted_rating, axis = 1)
df_movie.head(5)


### Popularity vs rating paradox

In [ ]:
sns.scatterplot(data=df_movie, x='popularity', y='vote_average',hue='adult')
plt.savefig('popularity_vs_rating_paradox(scatterPlot).png', dpi=150)
plt.show()

# **FEATURE ENGINEERING / AUGMENTATION**

### Extract year/month/day

In [ ]:
df_movie['year'] = pd.to_datetime(df_movie['release_date']).dt.year
df_movie['month'] = pd.to_datetime(df_movie['release_date']).dt.month
df_movie['day'] = pd.to_datetime(df_movie['release_date']).dt.day
df_movie.head(2)

### Popularity categories

In [ ]:
df_movie['popularity_level'] = pd.cut(df_movie['popularity'],
                                      bins=[0,10,50,100,1000],
                                      labels=['low','medium','high','viral'])
df_movie.head(5)


### Rating categories

In [ ]:
df_movie['rating+category'] = pd.cut(df_movie['vote_average'],
                                     bins = [0,4,6,8,10],
                                     labels=['bad','average','good','excellent'])
df_movie.head(5)

## Key Insights

* The dataset follows a power law distribution — majority of movies are low popularity/engagement while a tiny fraction dominate

* vote_count acts as a reliability filter — movies with few votes have extreme/unreliable ratings (0–10), while heavily voted movies converge around 6–7

* The most popular movies are NOT the highest rated — popularity is driven by marketing and mass appeal, not quality. Niche films with small audiences often rate higher

* Adult movies represent a tiny minority — clustered at low popularity and low vote counts, confirming mainstream movies dominate the platform

* Raw vote_average is misleading — IMDb-style weighted rating gives fairer rankings by balancing a movie's own rating against the global mean

* Extracting year/month/day and creating popularity_level / rating_category buckets transforms raw numbers into actionable segments for analysis